In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class ConvBlock2(nn.Module):
  def __init__(self,in_channels,out_channels):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU()
    )

  def forward(self,x):
    return self.conv(x)



class ConvBlock3(nn.Module):
  def __init__(self,in_channels,out_channels):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1),
        nn.ReLU()
    )
  def forward(self, x):
    return self.conv(x)


In [ ]:
import torch
import torch.nn as nn

class VGG16(nn.Module):
    def __init__(self, n_channels, num_classes):
        super().__init__()

        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bloklar
        self.block1 = ConvBlock2(n_channels, 64)
        self.block2 = ConvBlock2(64, 128)
        self.block3 = ConvBlock3(128, 256)
        self.block4 = ConvBlock3(256, 512)
        self.block5 = ConvBlock3(512, 512)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes)
        )


    def forward(self, x):
        x = self.block1(x)
        x = self.max_pool(x)

        x = self.block2(x)
        x = self.max_pool(x)

        x = self.block3(x)
        x = self.max_pool(x)

        x = self.block4(x)
        x = self.max_pool(x)

        x = self.block5(x)
        x = self.max_pool(x)

        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

In [ ]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [ ]:
def train_fn(model, criterion, optimizer, train_loader, n_epochs, device):
    history = {
        'train_loss': [],
        'train_acc': []
    }

    for epoch in range(n_epochs):
        train_losses = 0.0
        train_accs = 0.0
        model.train()

        current_lr = optimizer.param_groups[0]['lr']

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
                preds = model(images)
                loss = criterion(preds, labels)

                _, predicted = torch.max(preds, 1)
                batch_acc = (predicted == labels).float().mean()


            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_losses += loss.item()
            train_accs += batch_acc.item()

        avg_train_loss = train_losses / len(train_loader)
        avg_train_acc = train_accs / len(train_loader)

        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(avg_train_acc)

        print(f"{{'Epoch: {epoch+1}/{n_epochs}| Train Loss: {avg_train_loss:.4f}| Train Acc: {avg_train_acc:.4f}| LR: {current_lr}'}}")

    return history

buraad cifar10 datasetini isledirem


In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

valid_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [ ]:
model = VGG16(n_channels=3, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


history = train_fn(model, criterion, optimizer, train_loader, n_epochs=10, device=device)

{'Epoch: 1/10| Train Loss: 2.3036| Train Acc: 0.0989| LR: 0.001'}
{'Epoch: 2/10| Train Loss: 2.3029| Train Acc: 0.0991| LR: 0.001'}
{'Epoch: 3/10| Train Loss: 2.3028| Train Acc: 0.0982| LR: 0.001'}
{'Epoch: 4/10| Train Loss: 2.3028| Train Acc: 0.0969| LR: 0.001'}
{'Epoch: 5/10| Train Loss: 2.3028| Train Acc: 0.0986| LR: 0.001'}
{'Epoch: 6/10| Train Loss: 2.3028| Train Acc: 0.0984| LR: 0.001'}
{'Epoch: 7/10| Train Loss: 2.3028| Train Acc: 0.0975| LR: 0.001'}
{'Epoch: 8/10| Train Loss: 2.3028| Train Acc: 0.0986| LR: 0.001'}
{'Epoch: 9/10| Train Loss: 2.3027| Train Acc: 0.0993| LR: 0.001'}
{'Epoch: 10/10| Train Loss: 2.3028| Train Acc: 0.0986| LR: 0.001'}
